In [8]:
"""
Main Engine v1.2 - Bayesian ELO as the primary ranking system

This is the updated top-level engine that uses the BayesianELORanking
implementation (with uncertainty tracking) as the ranking system for
alpha and tail-survival tournaments.

Drop this file next to your existing modules (data_manager, regime_detector,
strategy_zoo, backtest_engine, matchmaker, ranking, analyzer) and run.
"""

from data_manager import DataManager
from regime_detector import HMMRegimeDetector 
from regime_ensemble import run_all_detectors
from strategy_zoo import StrategyRegistry, BuyAndHold, SMACrossover, BollingerMeanReversion, RSIMomentum
from backtest_engine import BacktestEngine, BacktestResult
from matchmaker import RoundRobinMatcher
from ranking import RankingManager, BayesianELORanking  # <- use Bayesian ranking
from analyzer import MetaELOAnalyzer
from itertools import combinations
from utils import align_series_to_index, sanity_check_market_data

import pandas as pd
import numpy as np
from typing import Dict, Optional

# New imports for safe snapshot & saving
import os
import glob
import math

# ---------------------------
# Helper: comparison function
# ---------------------------
def _compare_strategies_alpha(
    result_a: BacktestResult,
    result_b: BacktestResult,
    metric: str = 'alpha'
) -> float:
    """
    Compare strategies on ALPHA (market-neutral returns), not raw returns.
    Returns:
        1.0 if A > B, 0.0 if B > A, 0.5 if tie.
    """
    if metric == 'alpha':
        # Compare on alpha returns if available
        if getattr(result_a, 'alpha_returns', None) is not None and getattr(result_b, 'alpha_returns', None) is not None:
            alpha_a = float(np.nanmean(result_a.alpha_returns))
            alpha_b = float(np.nanmean(result_b.alpha_returns))
        else:
            # Fallback to raw returns (not ideal but safe)
            alpha_a = float(np.nanmean(result_a.returns))
            alpha_b = float(np.nanmean(result_b.returns))

    elif metric == 'tail_survival':
        tail_a = getattr(result_a, 'tail_metrics', {}) or {}
        tail_b = getattr(result_b, 'tail_metrics', {}) or {}

        # We define "survival score" as stress_mean (higher is better) - choose robust fallback
        alpha_a = float(tail_a.get('stress_mean', -np.inf))
        alpha_b = float(tail_b.get('stress_mean', -np.inf))

    else:
        # Generic metric access (from result.metrics)
        alpha_a = float(result_a.metrics.get(metric, -np.inf))
        alpha_b = float(result_b.metrics.get(metric, -np.inf))

    # Comparison with tolerance
    if np.isnan(alpha_a) and np.isnan(alpha_b):
        return 0.5
    if np.isnan(alpha_a):
        return 0.0
    if np.isnan(alpha_b):
        return 1.0

    if alpha_a > alpha_b:
        return 1.0
    elif alpha_b > alpha_a:
        return 0.0
    else:
        return 0.5

# ---------------------------
# Rating history snapshot + safe saving
# ---------------------------
# Buffer and config
rating_history = []  # list of dicts
FLUSH_EVERY = 250  # flush every N snapshots to parquet chunks
_out_chunk_dir = 'rating_history_chunks'
os.makedirs(_out_chunk_dir, exist_ok=True)
_flush_counter = 0

def _safe_iter_players(ranking_obj):
    """Return iterable of (name, player) pairs for several possible ranking implementations."""
    if ranking_obj is None:
        return []
    # Common: ranking_obj.players is a dict
    if hasattr(ranking_obj, 'players') and isinstance(ranking_obj.players, dict):
        return ranking_obj.players.items()
    # Common alternative: ranking_obj.get_players() -> dict or list
    if hasattr(ranking_obj, 'get_players'):
        try:
            players = ranking_obj.get_players()
            if isinstance(players, dict):
                return players.items()
            if isinstance(players, (list, tuple)):
                out = []
                for p in players:
                    name = getattr(p, 'name', getattr(p, 'strategy', None))
                    if name is None:
                        continue
                    out.append((name, p))
                return out
        except Exception:
            pass
    # fallback: 'ratings' attribute
    if hasattr(ranking_obj, 'ratings') and isinstance(ranking_obj.ratings, dict):
        return ranking_obj.ratings.items()
    return []

def snapshot_ratings(rm, metric, regime, ts):
    """Append ratings snapshot for metric/regime at timestamp ts (pd.Timestamp or convertible)."""
    global _flush_counter
    ts = pd.Timestamp(ts)
    ranking_obj = rm.rankings.get(metric, {}).get(regime)
    if ranking_obj is None:
        return

    for name, player in _safe_iter_players(ranking_obj):
        try:
            mu = float(getattr(player, 'mu', getattr(player, 'rating', float('nan'))))
        except Exception:
            mu = float('nan')
        try:
            sigma = float(getattr(player, 'sigma', getattr(player, 'uncertainty', float('nan'))))
        except Exception:
            sigma = float('nan')
        # optional fields if present
        try:
            conservative = float(getattr(player, 'conservative_rating', math.nan))
        except Exception:
            conservative = math.nan
        try:
            games = int(getattr(player, 'games', getattr(player, 'matches', math.nan)))
        except Exception:
            games = None

        rating_history.append({
            'date': ts,
            'metric': metric,
            'regime': regime,
            'strategy': name,
            'mu': mu,
            'sigma': sigma,
            'conservative_rating': conservative,
            'games': games
        })

    _flush_counter += 1
    # Periodic flush to parquet to avoid memory spikes
    if _flush_counter >= FLUSH_EVERY:
        _flush_counter = 0
        df_chunk = pd.DataFrame(rating_history)
        if not df_chunk.empty:
            fname = os.path.join(_out_chunk_dir, f'rating_chunk_{pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")}.parquet')
            df_chunk.to_parquet(fname, index=False, compression='gzip')
            rating_history.clear()

def save_final_rating_history(out_filename='rating_history.parquet'):
    """Combine chunks and in-memory buffer, write a single parquet file."""
    frames = []
    chunk_files = sorted(glob.glob(os.path.join(_out_chunk_dir, '*.parquet')))
    for cf in chunk_files:
        try:
            frames.append(pd.read_parquet(cf))
        except Exception:
            try:
                frames.append(pd.read_pickle(cf))
            except Exception:
                pass
    if rating_history:
        frames.append(pd.DataFrame(rating_history))
    if not frames:
        print("No rating history to save.")
        return
    df_all = pd.concat(frames, ignore_index=True)
    df_all = df_all.sort_values(['date','metric','regime','strategy']).reset_index(drop=True)
    df_all.to_parquet(out_filename, index=False, compression='gzip')
    print(f"Saved final rating history to {out_filename} (rows={len(df_all)})")

# ---------------------------
# Main run loop
# ---------------------------
def run():
    print("="*100)
    print("Trading Strategy ELO Engine v1.2 - Bayesian ELO (with Uncertainty) + Tail Risk")
    print("="*100)

    # Initialize components
    print("\n[Step 1] Initializing components...")
    data_manager = DataManager(cache_dir='./data_cache')
    regime_detector = HMMRegimeDetector(num_regimes=3)
    strategy_registry = StrategyRegistry()

    # Backtest engine (expects to support benchmark_data and to populate alpha/beta on results)
    backtest_engine = BacktestEngine(benchmark_symbol='SPY')

    # Matchmaker: round-robin across strategies and regimes
    matchmaker = RoundRobinMatcher(include_global=True)

    # Use the BayesianELORanking as the ranking backend for the RankingManager.
    ranking_manager = RankingManager(
        ranking_class=BayesianELORanking,
        initial_mu=1500,
        initial_sigma=350,
        min_sigma=50,
        tau=1.0,
        base_k=32
    )

    # Register strategies (simple set for demo)
    print("\n[Step 2] Registering strategies...")
    strategy_registry.register(BuyAndHold())
    strategy_registry.register(SMACrossover(fast_period=20, slow_period=50))
    strategy_registry.register(SMACrossover(fast_period=50, slow_period=200))
    strategy_registry.register(BollingerMeanReversion(period=20, num_std=2.0))
    strategy_registry.register(RSIMomentum(period=14))

    # Experiment parameters
    target_symbol = 'SPY'
    benchmark_symbol = 'SPY'
    start_date = '2018-01-01'
    end_date = '2024-01-01'
    initial_capital = 100_000

    try:
        # Fetch market and benchmark data
        print(f"\n[Step 3] Fetching market data for {target_symbol}...")
        market_data = data_manager.fetch_data(symbol=target_symbol, start_date=start_date, end_date=end_date)

        sanity_check_market_data(market_data, require_cols=['close', 'volume'], nan_threshold=0.05)

        # For now benchmark = target; production could fetch separately
        benchmark_data = market_data.copy()

        # Detect regimes
        print("\n[Step 4] Detecting regimes...")
        regimes_df = run_all_detectors(market_data, hmm_bootstrap_iters=80, hmm_block_size=5, verbose=True)
        print(regimes_df)
        #export to csv
        regimes_df.to_csv('regimes.csv')
        regimes = regime_detector.detect_regimes(market_data)
        regimes = align_series_to_index(regimes, market_data.index, method='ffill')
        #regimes.to_frame(name='regime').reset_index().to_csv('regimes.csv', index=False)
        # regimes should be a pd.Series indexed the same as market_data

        # Run backtests with benchmark data - populate dictionary of results
        print("\n[Step 5] Running backtests (alpha/beta separation enabled in engine)...")
        all_results: Dict[str, BacktestResult] = {}
        strategies_list = [s for s in strategy_registry.strategies.values()]

        for strategy in strategies_list:
            print(f"   Backtesting {strategy.name} ...", end='', flush=True)
            result: BacktestResult = backtest_engine.run_backtest(
                strategy,
                market_data,
                initial_capital=initial_capital,
                benchmark_data=benchmark_data  # engine should compute alpha/beta if implemented
            )
            all_results[strategy.name] = result

            try:
                # Reindex to the master calendar, then fill any new missing values with 0
                result.returns = result.returns.reindex(market_data.index).fillna(0)
            except Exception:
                # keep original if alignment fails but log
                print(f"Warning: failed to align returns for {strategy.name}")

            if getattr(result, 'alpha_returns', None) is not None:
                try:
                    # Do the same for alpha returns
                    result.alpha_returns = result.alpha_returns.reindex(market_data.index).fillna(0)
                except Exception:
                    print(f"Warning: failed to align alpha_returns for {strategy.name}")    

            print(" done.")

            # Print quick diagnostics when available
            beta_str = f"{result.beta:6.3f}" if getattr(result, 'beta', None) is not None else "  n/a "
            tail_metrics = getattr(result, 'tail_metrics', {}) or {}
            stress_mean = tail_metrics.get('stress_mean', np.nan)
            stress_days = tail_metrics.get('stress_positive_days', None)
            stress_days_str = f"{int(stress_days):2d}" if stress_days is not None else " n/a"
            print(f"      Beta: {beta_str} | StressMean: {stress_mean:7.4f} | StressDays: {stress_days_str}")

        # Generate matchups (informational - keep for compatibility)
        print("\n[Step 6] Generating round-robin matchups (per regime + global)...")
        matchups = matchmaker.generate_matchups(strategies_list, market_data, regimes, asset=target_symbol)

        # -------------------------------
        # NEW: Run daily tournaments
        # -------------------------------
        print("\n[Step 7] Running DAILY Bayesian ELO tournaments (alpha preferred)...")

        # --- compute stress days from benchmark returns (e.g., worst 5% days)
        benchmark_ret = benchmark_data['close'].pct_change().dropna()
        stress_q = 0.05
        stress_threshold = benchmark_ret.quantile(stress_q)   # e.g., 5th percentile
        stress_mask = benchmark_ret <= stress_threshold       # Series indexed by date: True on stress days

        # Optionally expand to full index if needed
        #stress_mask = stress_mask.reindex(market_data.index).fillna(False)
        stress_mask = stress_mask.reindex(market_data.index).fillna(False).astype(bool)


        # Build evaluation dates: intersection of available dates across strategies (alpha preferred)
        all_dates = None
        for res in all_results.values():
            if getattr(res, 'alpha_returns', None) is not None:
                idx = res.alpha_returns.dropna().index
            else:
                idx = res.returns.dropna().index
            if all_dates is None:
                all_dates = set(idx)
            else:
                all_dates = all_dates.intersection(set(idx))

        if not all_dates:
            raise RuntimeError("No overlapping evaluation dates found across strategy results.")

        eval_dates = sorted(list(all_dates))
        print(f"  Evaluation dates (count): {len(eval_dates)}")

        # Precompute pairs
        pairs = list(combinations(strategies_list, 2))

        match_counter = 0
        # Loop through dates and update ranking per pair per day
        for date in eval_dates:
            regime_label = None
            try:
                if regimes is not None and date in regimes.index:
                    regime_label = regimes.loc[date]
            except Exception:
                regime_label = None

            # Build daily scores (alpha preferred)
            scores = {}
            for strat in strategies_list:
                res = all_results[strat.name]
                score = np.nan
                if getattr(res, 'alpha_returns', None) is not None:
                    try:
                        score = float(res.alpha_returns.loc[date])
                    except Exception:
                        score = np.nan
                if np.isnan(score):
                    try:
                        score = float(res.returns.loc[date])
                    except Exception:
                        score = np.nan
                scores[strat.name] = score

            # Pairwise comparisons
            for strat_a, strat_b in pairs:
                sa = scores.get(strat_a.name, np.nan)
                sb = scores.get(strat_b.name, np.nan)

                if np.isnan(sa) or np.isnan(sb):
                    # skip incomplete data for this date
                    continue

                eps = 1e-12
                if sa > sb + eps:
                    outcome = 1.0
                elif sb > sa + eps:
                    outcome = 0.0
                else:
                    outcome = 0.5

                ts = pd.Timestamp(date)
                ranking_manager.update(
                    strategy_a_name=strat_a.name,
                    strategy_b_name=strat_b.name,
                    outcome=outcome,
                    metric='alpha',
                    regime=regime_label,
                    timestamp=ts
                )

                # Also update GLOBAL (regime=None) so we accumulate an overall rating.
                # This keeps per-regime learning AND a running global leaderboard.
                ranking_manager.update(
                    strategy_a_name=strat_a.name,
                    strategy_b_name=strat_b.name,
                    outcome=outcome,
                    metric='alpha',
                    regime=None,
                    timestamp=ts
                )

                match_counter += 1

            snapshot_ratings(ranking_manager, 'alpha', regime_label, ts)
            snapshot_ratings(ranking_manager, 'alpha', None, ts)
            # also snapshot tail_survival if you want
            snapshot_ratings(ranking_manager, 'tail_survival', regime_label, ts)
            snapshot_ratings(ranking_manager, 'tail_survival', None, ts)

            # -----------------------------
            # Per-day tail_survival updates
            # -----------------------------
            # If today is a market stress day, reward strategies that survived (positive returns).
            is_stress = bool(stress_mask.loc[date]) if date in stress_mask.index else False

            if is_stress:
                # build tail_scores dict same shape as 'scores'
                tail_scores = {}
                for strat in strategies_list:
                    res = all_results[strat.name]
                    s_val = np.nan
                    if getattr(res, 'alpha_returns', None) is not None:
                        try:
                            s_val = float(res.alpha_returns.loc[date])
                        except Exception:
                            s_val = np.nan
                    if np.isnan(s_val):
                        try:
                            s_val = float(res.returns.loc[date])
                        except Exception:
                            s_val = np.nan
                    tail_scores[strat.name] = s_val

                # pairwise tail updates (reward positive survival on stress day)
                for strat_a, strat_b in pairs:
                    sa = tail_scores.get(strat_a.name, np.nan)
                    sb = tail_scores.get(strat_b.name, np.nan)
                    # skip if either missing
                    if np.isnan(sa) or np.isnan(sb):
                        continue

                    # Outcome logic: prefer strategy with higher return on stress day.
                    eps = 1e-12
                    if sa > sb + eps:
                        outcome = 1.0
                    elif sb > sa + eps:
                        outcome = 0.0
                    else:
                        outcome = 0.5

                    ts = pd.Timestamp(date)
                    ranking_manager.update(
                        strategy_a_name=strat_a.name,
                        strategy_b_name=strat_b.name,
                        outcome=outcome,
                        metric='tail_survival',
                        regime=regime_label,   # keep regime-specific tail learning
                        timestamp=ts
                    )

                    # also update global tail_survival
                    ranking_manager.update(
                        strategy_a_name=strat_a.name,
                        strategy_b_name=strat_b.name,
                        outcome=outcome,
                        metric='tail_survival',
                        regime=None,
                        timestamp=ts
                    )

        print(f"\nDaily tournament complete. Total daily matches processed: {match_counter}")

        # Save rating history (final) before printing final leaderboards
        save_final_rating_history('rating_history.parquet')

        # Final leaderboards: use BayesianELORanking's uncertainty-aware leaderboard where available
        print("\n" + "="*100)
        print("FINAL BAYESIAN ELO LEADERBOARDS (Alpha & Tail Survival)")
        print("="*100)

        # Alpha (global)
        print("\n--- BAYESIAN ALPHA LEADERBOARD (global) ---")
        # RankingManager stores ranking objects in rankings[metric][regime]
        alpha_global_ranking_system = ranking_manager.rankings.get('alpha', {}).get(None)
        if alpha_global_ranking_system is not None and hasattr(alpha_global_ranking_system, 'get_leaderboard_with_uncertainty'):
            print(alpha_global_ranking_system.get_leaderboard_with_uncertainty(sort_by='conservative_rating', top_n=None).to_string(index=True))
        else:
            print(ranking_manager.get_leaderboard(metric='alpha', regime=None).to_string())

        # Tail survival (global)
        print("\n--- BAYESIAN TAIL SURVIVAL LEADERBOARD (global) ---")
        tail_global_ranking_system = ranking_manager.rankings.get('tail_survival', {}).get(None)
        if tail_global_ranking_system is not None and hasattr(tail_global_ranking_system, 'get_leaderboard_with_uncertainty'):
            print(tail_global_ranking_system.get_leaderboard_with_uncertainty(sort_by='conservative_rating', top_n=None).to_string(index=True))
        else:
            print(ranking_manager.get_leaderboard(metric='tail_survival', regime=None).to_string())

        # Per-regime leaderboards (alpha)
        regimes_seen = sorted(list(regimes.unique())) if hasattr(regimes, 'unique') else []
        for regime_label in regimes_seen:
            print(f"\n--- BAYESIAN ALPHA LEADERBOARD (Regime {regime_label}) ---")
            ranking_sys = ranking_manager.rankings.get('alpha', {}).get(regime_label)
            if ranking_sys is not None and hasattr(ranking_sys, 'get_leaderboard_with_uncertainty'):
                print(ranking_sys.get_leaderboard_with_uncertainty(sort_by='conservative_rating').to_string(index=True))
            else:
                df = ranking_manager.get_leaderboard(metric='alpha', regime=regime_label)
                if not df.empty:
                    print(df.to_string())
                else:
                    print(" (no data) ")

        # Meta-analysis using MetaELOAnalyzer (works over RankingManager)
        print("\n" + "="*100)
        print("META ANALYSIS (Bayesian ELO)")
        print("="*100)

        meta_analyzer = MetaELOAnalyzer(ranking_manager)

        print("\n--- ELO VELOCITY (Alpha metric, global) ---")
        velocity = meta_analyzer.get_velocity_ranking(metric='alpha', regime=None)
        if isinstance(velocity, pd.DataFrame) and not velocity.empty:
            print(velocity.to_string())
        else:
            print("(no velocity data)")

        print("\n--- STRATEGY LIFECYCLE (Alpha metric, global) ---")
        lifecycle = meta_analyzer.get_lifecycle_report(metric='alpha', regime=None)
        if isinstance(lifecycle, pd.DataFrame) and not lifecycle.empty:
            print(lifecycle.to_string())
        else:
            print("(no lifecycle data)")

    except Exception as e:
        print(f"\n[ERROR] {e}")
        import traceback
        traceback.print_exc()

        # Save partial rating history on error too
        try:
            save_final_rating_history('rating_history_partial_on_error.parquet')
        except Exception:
            # fallback: pickle small buffer if parquet fails
            df_rating_history = pd.DataFrame(rating_history)
            df_rating_history.to_pickle('rating_history_error.pkl')

if __name__ == "__main__":
    run()


Trading Strategy ELO Engine v1.2 - Bayesian ELO (with Uncertainty) + Tail Risk

[Step 1] Initializing components...

[Step 2] Registering strategies...

[Step 3] Fetching market data for SPY...
Loading SPY from cache...

[Step 4] Detecting regimes...
[vol] done, regimes: [np.int64(0), np.int64(1), np.int64(2)]


/home/t/Downloads/arena/regime_detector.py:144: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  expanded = expanded.fillna(method='ffill').fillna(method='bfill')
INFO:regime_detector:HMM fitted: best_score=4699.210, cov_type=full
/home/t/Downloads/arena/regime_detector.py:144: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  expanded = expanded.fillna(method='ffill').fillna(method='bfill')
INFO:regime_detector:HMM fitted: best_score=-4047.998, cov_type=tied
/home/t/Downloads/arena/regime_ensemble.py:157: UserWarning: ChangePoint detector failed: ruptures not installed. Install with: pip install ruptures
  warnings.warn(f"ChangePoint detector failed: {e}")


[hmm] done, entropy and bootstrap stability added.


/home/t/Downloads/arena/regime_detector.py:144: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  expanded = expanded.fillna(method='ffill').fillna(method='bfill')


[mf] done.
Saved regimes_all.parquet with columns: ['regime_vol', 'regime_hmm', 'hmm_entropy', 'hmm_boot_agree', 'regime_cp', 'cp_break', 'regime_mf', 'consensus_regime', 'consensus_strength', 'num_detectors']
                           regime_vol  regime_hmm  hmm_entropy  \
Date                                                             
2018-01-02 00:00:00-05:00           0           0          NaN   
2018-01-03 00:00:00-05:00           0           0          NaN   
2018-01-04 00:00:00-05:00           0           1          NaN   
2018-01-05 00:00:00-05:00           0           1          NaN   
2018-01-08 00:00:00-05:00           0           1          NaN   
...                               ...         ...          ...   
2023-12-22 00:00:00-05:00           0           1          NaN   
2023-12-26 00:00:00-05:00           0           1          NaN   
2023-12-27 00:00:00-05:00           0           1          NaN   
2023-12-28 00:00:00-05:00           0           1          NaN  

INFO:regime_detector:HMM fitted: best_score=4699.210, cov_type=full



[Step 5] Running backtests (alpha/beta separation enabled in engine)...
   Backtesting BuyAndHold ... done.
      Beta:  0.819 | StressMean: -0.0255 | StressDays:  0
   Backtesting SMAXover_20_50 ... done.
      Beta: -0.231 | StressMean:  0.0052 | StressDays: 40
   Backtesting SMAXover_50_200 ... done.
      Beta:  0.315 | StressMean: -0.0052 | StressDays: 31
   Backtesting BollingerMR_20_2.0 ... done.
      Beta:  0.450 | StressMean: -0.0152 | StressDays: 22
   Backtesting RSI_14 ... done.
      Beta:  0.134 | StressMean: -0.0042 | StressDays: 24

[Step 6] Generating round-robin matchups (per regime + global)...

[Step 7] Running DAILY Bayesian ELO tournaments (alpha preferred)...
  Evaluation dates (count): 1509


/tmp/ipykernel_26219/3158804388.py:313: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  stress_mask = stress_mask.reindex(market_data.index).fillna(False).astype(bool)



Daily tournament complete. Total daily matches processed: 15090
Saved final rating history to rating_history.parquet (rows=20885)

FINAL BAYESIAN ELO LEADERBOARDS (Alpha & Tail Survival)

--- BAYESIAN ALPHA LEADERBOARD (global) ---
             strategy           mu      sigma  conservative_rating  signal_to_noise  games
1      SMAXover_20_50  1638.101248  63.075911          1511.949426        25.970315   6036
2     SMAXover_50_200  1570.394622  63.075911          1444.242800        24.896900   6036
3          BuyAndHold  1504.282674  63.075911          1378.130852        23.848767   6036
4              RSI_14  1431.443381  63.075911          1305.291559        22.693979   6036
5  BollingerMR_20_2.0  1355.778075  63.075911          1229.626253        21.494388   6036

--- BAYESIAN TAIL SURVIVAL LEADERBOARD (global) ---
             strategy           mu     sigma  conservative_rating  signal_to_noise  games
1     SMAXover_50_200  1729.514139  63.07593          1603.362278        27.41

In [1]:
import pandas as pd
master = pd.read_parquet("reports/master.parquet")
per_regime = pd.read_csv("reports/per_regime_metrics.csv")
global_df = pd.read_csv("reports/global_metrics.csv")
pred = pd.read_csv("reports/regime_predictiveness.csv").iloc[0]['predictiveness']
print("Predictiveness:", pred)
display(per_regime.sort_values(['strategy','regime']).head(20))
# Quick pivot: strategies (rows) x regime (cols) by Sharpe
pivot = per_regime.pivot(index='strategy', columns='regime', values='sharpe')
display(pivot)


Predictiveness: 0.5666666666666667


,strategy,regime,n_days,mean_daily_return,annual_return,annual_vol,sharpe,win_rate,max_drawdown
0,BollingerMR_20_2.0,0,2258,0.020700,173.724967,15.723543,11.048716,0.421169,-1.025860
1,BollingerMR_20_2.0,1,236,-0.003364,-0.572226,0.573882,-0.997115,0.584746,-0.668170
2,BollingerMR_20_2.0,2,22,-0.014375,-0.973977,0.916631,-1.062562,0.318182,-0.399435
3,BuyAndHold,0,2258,0.000964,0.274832,0.111959,2.454757,0.607174,-0.210756
4,BuyAndHold,1,236,-0.004499,-0.678986,0.063706,-10.658048,0.000000,-0.654845
5,BuyAndHold,2,22,-0.010708,-0.933662,0.269542,-3.463882,0.136364,-0.208389
6,RSI_14,0,2258,-0.000181,-0.044660,0.078743,-0.567160,0.408326,-0.371387
7,RSI_14,1,236,-0.000089,-0.022235,0.060049,-0.370286,0.516949,-0.057078
8,RSI_14,2,22,-0.002939,-0.523677,0.267889,-1.954831,0.272727,-0.072099
9,SMAXover_20_50,0,2258,0.000041,0.010483,0.166687,0.062892,0.516386,-0.445329


regime,0,1,2
strategy,,,
BollingerMR_20_2.0,11.048716,-0.997115,-1.062562
BuyAndHold,2.454757,-10.658048,-3.463882
RSI_14,-0.567160,-0.370286,-1.954831
SMAXover_20_50,0.062892,0.415998,-1.814663
SMAXover_50_200,0.544460,-2.177411,-0.637575


In [1]:
# Quick analysis script to run after your main script finishes

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- CONFIG ---
PERIOD_TO_ANALYZE = 'period_2' # Change this to 'period_1' or 'period_3'
METHOD = 'hmm' # Or 'gmm', 'kmeans'
# ---

# 1. Load the features and the regime labels
df_full = pd.read_csv('rigorous_regime_results/feature_data_for_analysis.csv', index_col=0, parse_dates=True) # You'll need to save this in your main script
regime_labels = pd.read_csv(f'rigorous_regime_results/{PERIOD_TO_ANALYZE}_{METHOD}_test_labels.csv', index_col=0, parse_dates=True, header=None, names=['date', 'regime'])

# 2. Merge them
df_merged = df_full.join(regime_labels, how='inner')

# 3. Calculate the average feature profile for each regime
regime_profiles = df_merged.groupby('regime').mean()

# Let's look at a few key features that define market stress
key_features = ['vol_63', 'mom_63', 'kurt_63', 'ret']
print(f"--- Average Feature Profiles for {PERIOD_TO_ANALYZE} ({METHOD}) ---")
print(regime_profiles[key_features])

# 4. Visualize it
plt.figure(figsize=(12, 8))
sns.heatmap(regime_profiles[key_features].T, annot=True, cmap='viridis', fmt='.3f')
plt.title(f'Mean Feature Values per Regime ({PERIOD_TO_ANALYZE.upper()}, {METHOD.upper()})')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'rigorous_regime_results/feature_data_for_analysis.csv'